### Your name:

<pre> Shamir Alavi</pre>

### Collaborators:

<pre> None </pre>


In [1]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

Open the housing data


In [2]:
import os
import tarfile

HOUSING_PATH = os.path.join('..', 'data')

def fetch_housing_data(housing_path=HOUSING_PATH):
    if not os.path.isdir(housing_path):
        os.makedirs(housing_path)
    tgz_path = os.path.join(housing_path, "housing.tgz")
    housing_tgz = tarfile.open(tgz_path)
    housing_tgz.extractall(path=housing_path)
    housing_tgz.close()

# fetch_housing_data()

import pandas as pd

def load_housing_data(housing_path=HOUSING_PATH):
    csv_path = os.path.join(housing_path, "housing.csv")
    return pd.read_csv(csv_path)

housing_original = load_housing_data()

### Build full pipeline for the data analysis following the example of the notebook.
 Hint: the main part requested to change is the algorithm used (KNN regression)


#### Considerations for building pipeline:

- Make your notebook as compact as possible. 
- Split data into training and testing sets below.
- Convert all categorical data to one-hot vectors below
- Normalize all non-categorical data 
-  Perform KNN regression using a variety of values for n_neighbors (K) between 1 and 10 and both "uniform" and "distance" weights via a grid search where  *housing_labels* is the output and all other features are the input (similar to as seen in lecture two.)

## Clean bad data
Looks like some median house values were capped to their nearest round numbers. These are data artifacts we want to remove to make the dataset cleaner. 

In [3]:
# Suggestion from the book
suspicious_values = [
    450000, 400000, 375000, 350000, 325000, 300000, 275000,
    250000, 225000, 200000, 187500, 175000, 162500, 150000,
    137500, 125000, 112500, 100000, 87500, 75000, 67500
]

housing = housing_original[~housing_original["median_house_value"].isin(suspicious_values)]

# Make sure there are no missing indices after removing rows
housing = housing.reset_index(drop=True)

### Regular train-test split with SimpleImputer (used for performance comparison)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

train_set, test_set = train_test_split(
    housing,
    test_size=0.2,
    random_state=42
)

housing_num = train_set.drop("ocean_proximity", axis=1)

imputer = SimpleImputer(strategy="median")

imputer.fit(housing_num)

X_train = imputer.transform(housing_num)

housing_tr = pd.DataFrame(X_train, columns=housing_num.columns.tolist(), index=train_set.index)

# add categorical column back to the training set
housing_tr["ocean_proximity"] = train_set["ocean_proximity"]

### Stratification

#### Stratified train-test split by household median income

In [4]:
# dataset is divided into 5 income categories
housing["income_cat"] = pd.cut(housing["median_income"], bins=[0.,1.5,3.0,4.5,6.,np.inf], labels=[1,2,3,4,5])

In [5]:
from sklearn.model_selection import StratifiedShuffleSplit

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index]
    strat_test_set = housing.loc[test_index]

### Handle missing values: Perform stratified imputation

In [6]:
training_set_medians = strat_train_set.groupby("income_cat")["total_bedrooms"].median()

# Apply medians learned from training set on both training and test sets
strat_train_set["total_bedrooms"] = strat_train_set.apply(
    lambda row: training_set_medians[row["income_cat"]] if pd.isna(row["total_bedrooms"]) else row["total_bedrooms"],
    axis=1
)

strat_test_set["total_bedrooms"] = strat_test_set.apply(
    lambda row: training_set_medians[row["income_cat"]] if pd.isna(row["total_bedrooms"]) else row["total_bedrooms"],
    axis=1
)

### Create new features

In [7]:
# bedrooms_ratio, rooms_per_bedroom (how many rooms are there for each bedroom)
strat_train_set["bedrooms_ratio"] = strat_train_set["total_bedrooms"] / strat_train_set["total_rooms"]
strat_train_set["rooms_per_bedroom"] = strat_train_set["total_rooms"] / strat_train_set["total_bedrooms"]

strat_test_set["bedrooms_ratio"] = strat_test_set["total_bedrooms"] / strat_test_set["total_rooms"]
strat_test_set["rooms_per_bedroom"] = strat_test_set["total_rooms"] / strat_test_set["total_bedrooms"]

##### Drop column used for stratification

In [8]:
strat_train_set = strat_train_set.drop('income_cat', axis=1)
strat_test_set = strat_test_set.drop('income_cat', axis=1)

## Training with Cross Validation
Create a pipeline to perform training with 10-fold cross validation.

The pipeline does the following:
1. Feature scaling:
    1. Log normalizes heavy-tailed features (log1p)
    2. Standardizes all features using StandardScaler
2. Feature engineering:
    1. Creates four (4) new features from the multimodally distributed housing_median_age using RBF kernel. There are roughly four peaks in the distribution at 15, 25, 35, and 50.
    2. Several gamma values for the kernel was tested before settling on 0.025. Check out section 'Create new features using RBF' to visualize the effects of different gamma values.
3. One-hot encoding of categorical feature ocean_proximity
4. Grid search on KNeighborsRegressor:
    1. Performs 10-fold cross validation.
    2. Searches across N = 1 to 10 and both "uniform" and "distance" weights.
5. Scoring: Uses neg_root_mean_squared_error to score model's performance across the 10 folds

In [9]:
import time
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer, OneHotEncoder
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV


# Features
log_cols = [
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income"
]

standard_cols = [
    "longitude",
    "latitude",
    "housing_median_age"
]

cat_cols = [
    "ocean_proximity"
]


# Log transformation + scaling
log_pipeline = Pipeline([
    ("log", FunctionTransformer(np.log1p)),
    ("scale", StandardScaler())
])


# RBF features
def rbf_features(X):
    return np.hstack([
        rbf_kernel(X, [[15]], gamma=0.025),
        rbf_kernel(X, [[25]], gamma=0.025),
        rbf_kernel(X, [[35]], gamma=0.025),
        rbf_kernel(X, [[50]], gamma=0.025)
    ])


rbf_pipeline = FunctionTransformer(rbf_features)


# Full preprocessing pipeline
preprocessing = ColumnTransformer([
    ("log", log_pipeline, log_cols),
    ("standard", StandardScaler(), standard_cols),
    ("rbf", rbf_pipeline, ["housing_median_age"]),
    ("cat", OneHotEncoder(), cat_cols)
])


# Full model pipeline
knn = Pipeline([
    ("preprocessing", preprocessing),
    ("regressor", KNeighborsRegressor())
])


# Grid search
param_grid = {
    "regressor__n_neighbors": range(1, 11),
    "regressor__weights": ["uniform", "distance"]
}


grid_search = GridSearchCV(
    knn,
    param_grid,
    cv=10,
    scoring="neg_root_mean_squared_error"
)

start = time.time()

# Run grid search on the training set after dropping target labels
grid_search.fit(strat_train_set.drop("median_house_value", axis=1),
                strat_train_set["median_house_value"])

end = time.time()

print(f"Time taken: {end - start:.2f} seconds")

print("Best parameters:", grid_search.best_params_)
print("Best CV RMSE:", -grid_search.best_score_)

Time taken: 15.40 seconds
Best parameters: {'regressor__n_neighbors': 10, 'regressor__weights': 'distance'}
Best CV RMSE: 59107.56832948242


### Check cross validation scores 

In [10]:
results = pd.DataFrame(grid_search.cv_results_)

results[
    [
        "param_regressor__n_neighbors",
        "param_regressor__weights",
        "mean_test_score",
        "std_test_score"
    ]
]

# convert scores to positive RMSE
results["mean_test_RMSE"] = -results["mean_test_score"]
results["std_test_RMSE"] = results["std_test_score"]

# display scores
results[
    [
        "param_regressor__n_neighbors",
        "param_regressor__weights",
        "mean_test_RMSE",
        "std_test_RMSE"
    ]
].sort_values("mean_test_RMSE")

,param_regressor__n_neighbors,param_regressor__weights,mean_test_RMSE,std_test_RMSE
19,10,distance,59107.568329,1375.985018
17,9,distance,59125.687095,1404.157768
15,8,distance,59269.883090,1508.375027
13,7,distance,59501.182383,1464.803536
11,6,distance,59726.553478,1452.308817
16,9,uniform,59782.522873,1384.426682
18,10,uniform,59785.731477,1366.691600
14,8,uniform,59907.012506,1527.514573
9,5,distance,60024.411992,1658.038566
12,7,uniform,60147.127601,1505.771846


## Evaluate on test set

In [ ]:
# Write your code here

### Conclusions
For what values of n_neighbors and weight does KNeighborsRegressor perform the best? Does it perform as well on the housing data as the linear regressor from the lectures? Why do you think this is?

<pre> WRITE RESPONSE HERE </pre>

### Read appending B

- Reflect on your last data project, read appendix B. Then, write down a few of the checklist items that your last data project could have used. If you have not yet done a data project, then write down a few of the items that you found most interesting.


### Submit your notebook

Submit your solution to Quercus
Make sure you rename your notebook to    
W2_UTORid.ipynb    
Example W2_adfasd01.ipynb
